In [ ]:
!pip install transformers datasets torch scikit-learn
!pip install --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 96.5 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


Testing Part

In [ ]:
import shutil
import os
from google.colab import files

#Save path
model_dir = "./results/final_model"


In [ ]:
# Define the path where the model was saved
model_path = "./results/final_model"

# Load the tokenizer that was used for training
# It's crucial to use the same tokenizer
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

# Load the trained model
# num_labels should be the same as when the model was trained
num_labels = train_df['label'].nunique() # Assuming train_df is still in scope
loaded_model = DistilBertForSequenceClassification.from_pretrained(model_path, num_labels=num_labels)

print(f"Model loaded from: {model_path}")

In [ ]:
# Define the path where the model was saved
model_path = "./results/final_model"

# Load the tokenizer that was used for training
# It's crucial to use the same tokenizer
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

# Load the trained model
# num_labels should be the same as when the model was trained
num_labels = train_df['label'].nunique() # Assuming train_df is still in scope
loaded_model = DistilBertForSequenceClassification.from_pretrained(model_path, num_labels=num_labels)

print(f"Model loaded from: {model_path}")

In [ ]:
import torch

# Example texts for prediction
new_texts = [
    "Congratulations! You've won a free vacation! Click here to claim your prize.",
    "Hi John, just wanted to confirm our meeting for tomorrow at 10 AM. See you there!",
    "Your order #12345 has been shipped. Track your package here."
]

# Map the numerical labels back to meaningful categories using the label_encoder
# Assuming label_encoder and PRIORITY_MAP are still in scope
# The inverse_transform needs a numpy array or list

# Get the unique priority values from the original PRIORITY_MAP
# Then sort them to match the sorted order that LabelEncoder would produce for map values.
# This assumes LabelEncoder encodes based on sorted unique values.
# Let's verify the mapping first.

# Create a mapping from encoded label to original priority value
encoded_to_priority = {encoded_label: priority_value for priority_value, encoded_label in zip(label_encoder.classes_, range(len(label_encoder.classes_)))}

# Create a mapping from priority value to unified label
priority_to_unified_label = {value: key for key, value in PRIORITY_MAP.items()}

def predict_priority_label(text, model, tokenizer, encoded_to_priority, priority_to_unified_label, max_len=128):
    model.eval() # Set model to evaluation mode
    inputs = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=max_len,
        return_tensors="pt"
    )

    with torch.no_grad():
        outputs = model(**inputs)

    # Get the predicted class (the one with the highest logit)
    predicted_label_id = torch.argmax(outputs.logits, dim=1).item()

    # Convert back to original priority value
    predicted_priority_value = encoded_to_priority[predicted_label_id]

    # Convert priority value to unified label
    predicted_unified_label = priority_to_unified_label[predicted_priority_value]

    return predicted_unified_label

print("Predictions:")
for text in new_texts:
    predicted_label = predict_priority_label(text, loaded_model, tokenizer, encoded_to_priority, priority_to_unified_label)
    print(f"Text: '{text[:60]}...' -> Predicted Category: {predicted_label}")

### Step 1: Data Enrichment with Sender and Application Context

Since our current datasets don't explicitly contain sender or application context, we'll simulate these features for demonstration purposes. In a real-world scenario, you would extract this information from your raw message data.

For `email_df`, we can assume 'email' as the application context and generate some example senders. For `sms_df`, we'll assume 'sms' and similarly generate senders. The 'sender' feature could represent a specific email address, a phone number, or a category of sender (e.g., 'known contact', 'unknown', 'system').

In [ ]:
import numpy as np

# Add 'application_context' to email_df
email_df['application_context'] = 'email'

# Simulate 'sender' for email_df
# Let's create a few common sender types and some random ones
email_senders = [
    'support@example.com', 'noreply@bank.com', 'marketing@shop.com',
    'friends@personal.com', 'work@company.org'
]
email_df['sender'] = np.random.choice(email_senders, size=len(email_df), p=[0.2, 0.2, 0.2, 0.2, 0.2])

# Add 'application_context' to sms_df
sms_df['application_context'] = 'sms'

# Simulate 'sender' for sms_df
sms_senders = [
    'Mom', 'BankAlert', 'FriendA', 'UnknownNumber', 'GoogleVerification'
]
sms_df['sender'] = np.random.choice(sms_senders, size=len(sms_df), p=[0.2, 0.2, 0.2, 0.2, 0.2])

print("Email DataFrame with new features:")
display(email_df.head())

print("\nSMS DataFrame with new features:")
display(sms_df.head())

Now that we have enriched our individual dataframes, we need to re-run the concatenation and train/test split. Then, we will need to update our `TextDataset` class to handle these new features and modify the model architecture to incorporate them into the classification task.

Training Part

In [ ]:
import pandas as pd
import torch
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments
)
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
from huggingface_hub import whoami, login
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)

whoami()

{'type': 'user',
 'id': '696b347b3adc65f5021a6f48',
 'name': 'Sai-Cheranjeeve',
 'fullname': 'Sai Cheranjeeve S',
 'isPro': False,
 'avatarUrl': 'https://cdn-avatars.huggingface.co/v1/production/uploads/no-auth/jCl613Bn9E04kk5F1xNww.png',
 'orgs': [],
 'auth': {'type': 'access_token',
  'accessToken': {'displayName': 'Mini-project',
   'role': 'fineGrained',
   'createdAt': '2026-03-12T02:33:14.394Z',
   'fineGrained': {'canReadGatedRepos': True,
    'global': [],
    'scoped': [{'entity': {'_id': '696b347b3adc65f5021a6f48',
       'type': 'user',
       'name': 'Sai-Cheranjeeve'},
      'permissions': []}]}}}}

In [ ]:
from datasets import load_dataset

email_ds = load_dataset("jason23322/high-accuracy-email-classifier")
email_df = email_ds["train"].to_pandas()
email_df = email_df.drop(columns=["id", "subject", "body","category_id"])
print(email_df.columns)




README.md: 0.00B [00:00, ?B/s]

train.json: 0.00B [00:00, ?B/s]

test.json: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Index(['text', 'category'], dtype='object')


In [ ]:


sms_ds = load_dataset("Deysi/spam-detection-dataset")
sms_df = sms_ds["train"].to_pandas()
print(sms_df.columns)


README.md:   0%|          | 0.00/581 [00:00<?, ?B/s]

data/train-00000-of-00001-daf190ce720b3d(…):   0%|          | 0.00/1.92M [00:00<?, ?B/s]

data/test-00000-of-00001-fa9b3e8ade89a33(…):   0%|          | 0.00/663k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8175 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2725 [00:00<?, ? examples/s]

Index(['text', 'label'], dtype='object')


In [ ]:
sms_df = sms_df.rename(columns={
    "text": "text",
    "label": "original_label"
})

email_df = email_df.rename(columns={
    "text": "text",
    "category": "original_label"
})

print(email_df.columns)
print(sms_df.columns)

Index(['text', 'original_label'], dtype='object')
Index(['text', 'original_label'], dtype='object')


In [ ]:
sms_df["source_dataset"] = "sms_dataset"
email_df["source_dataset"] = "email_dataset"


In [ ]:
def map_spam_label(label):
    if label.lower() == "spam":
        return "SPAM"
    else:
        return "PERSONAL"


In [ ]:
sms_df["unified_label"] = sms_df["original_label"].apply(map_spam_label)


In [ ]:
def map_category_label(label):
    label = label.lower()


    if label in ["updates","verify_code"]:
        return "UPDATES"
    elif label in ["social","forum"]:
        return "SOCIAL"
    elif label in ["marketing", "promotions"]:
        return "MARKETING"
    elif label in ["spam"]:
        return "SPAM"
    else:
        return "PERSONAL"


In [ ]:
email_df["unified_label"] = email_df["original_label"].apply(map_category_label)


In [ ]:
PRIORITY_MAP = {
    "UPDATES": 4,
    "PERSONAL": 3,   # highest
    "SOCIAL": 3,
    "MARKETING": 2,
    "SPAM": 1        # lowest
}


In [ ]:
sms_df["priority"] = sms_df["unified_label"].map(PRIORITY_MAP)
email_df["priority"] = email_df["unified_label"].map(PRIORITY_MAP)


In [ ]:
intermediate_df = pd.concat(
    [sms_df, email_df],
    ignore_index=True
)

print(intermediate_df.columns)


Index(['text', 'original_label', 'source_dataset', 'unified_label',
       'priority'],
      dtype='object')


In [ ]:

train_df, test_df = train_test_split(
    intermediate_df,
    test_size=0.2,
    random_state=42,
    stratify=intermediate_df['priority']   # important
)


In [ ]:
train_df = train_df[['text', 'priority', 'application_context', 'sender']].dropna()
test_df  = test_df[['text', 'priority', 'application_context', 'sender']].dropna()

NameError: name 'train_df' is not defined

In [ ]:
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)


In [ ]:


label_encoder = LabelEncoder()

train_df['label'] = label_encoder.fit_transform(train_df['priority'])
test_df['label']  = label_encoder.transform(test_df['priority'])


In [ ]:


tokenizer = DistilBertTokenizerFast.from_pretrained(
    "distilbert-base-uncased"
)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
import torch
from torch.utils.data import Dataset
from sklearn.preprocessing import LabelEncoder

class TextDataset(Dataset):
    def __init__(self, texts, labels, application_contexts, senders, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.application_contexts = application_contexts
        self.senders = senders
        self.tokenizer = tokenizer
        self.max_len = max_len

        # Encode application context and sender
        self.app_encoder = LabelEncoder()
        self.sender_encoder = LabelEncoder()

        self.encoded_applications = self.app_encoder.fit_transform(self.application_contexts)
        self.encoded_senders = self.sender_encoder.fit_transform(self.senders)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
            "application_context_id": torch.tensor(self.encoded_applications[idx], dtype=torch.long),
            "sender_id": torch.tensor(self.encoded_senders[idx], dtype=torch.long)
        }

In [ ]:
train_dataset = TextDataset(
    train_df['text'].tolist(),
    train_df['label'].tolist(),
    train_df['application_context'].tolist(),
    train_df['sender'].tolist(),
    tokenizer
)

test_dataset = TextDataset(
    test_df['text'].tolist(),
    test_df['label'].tolist(),
    test_df['application_context'].tolist(),
    test_df['sender'].tolist(),
    tokenizer
)

NameError: name 'train_df' is not defined

In [ ]:
from transformers import DistilBertForSequenceClassification, DistilBertModel
import torch.nn as nn

# Get the number of unique application contexts and senders from the dataset
# Assuming train_dataset is already initialized and has app_encoder and sender_encoder
num_application_contexts = len(train_dataset.app_encoder.classes_)
num_senders = len(train_dataset.sender_encoder.classes_)

class CustomDistilBertForSequenceClassification(DistilBertForSequenceClassification):
    def __init__(self, config):
        super().__init__(config)
        # Load the base DistilBERT model
        self.distilbert = DistilBertModel(config)
        self.pre_classifier = nn.Linear(config.dim, config.dim)

        # Define embedding layers for application context and sender
        self.application_context_embedding = nn.Embedding(num_application_contexts, config.dim // 4) # Smaller embedding dimension
        self.sender_embedding = nn.Embedding(num_senders, config.dim // 4) # Smaller embedding dimension

        # Adjust the classifier layer to take into account the new embeddings
        # Original: config.dim
        # New: config.dim (from DistilBERT) + config.dim // 4 (app) + config.dim // 4 (sender)
        # Using an intermediate hidden layer with ReLU activation
        combined_dim = config.dim + (config.dim // 4) + (config.dim // 4)

        self.classifier = nn.Sequential(
            nn.Linear(combined_dim, config.dim), # Intermediate layer
            nn.ReLU(),
            nn.Dropout(config.seq_classif_dropout),
            nn.Linear(config.dim, config.num_labels)
        )

        self.init_weights()

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        head_mask=None,
        inputs_embeds=None,
        labels=None,
        application_context_id=None,
        sender_id=None,
        output_attentions=None,
        output_hidden_states=None,
        return_dict=None,
    ):
        distilbert_output = self.distilbert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            head_mask=head_mask,
            inputs_embeds=inputs_embeds,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )

        hidden_state = distilbert_output[0]  # (bs, seq_len, dim)
        pooled_output = hidden_state[:, 0]  # (bs, dim) - Take the [CLS] token's hidden state
        pooled_output = self.pre_classifier(pooled_output)  # (bs, dim)
        pooled_output = nn.ReLU()(pooled_output)  # (bs, dim)

        # Get embeddings for application context and sender
        app_embeds = self.application_context_embedding(application_context_id) # (bs, embed_dim_app)
        sender_embeds = self.sender_embedding(sender_id) # (bs, embed_dim_sender)

        # Concatenate DistilBERT output with new embeddings
        combined_features = torch.cat((pooled_output, app_embeds, sender_embeds), dim=1)

        # Pass combined features through the new classifier
        logits = self.classifier(combined_features)

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))

        if not return_dict:
            output = (logits,) + distilbert_output[1:]
            return ((loss,) + output) if loss is not None else output

        return {
            "loss": loss,
            "logits": logits,
            "hidden_states": distilbert_output.hidden_states,
            "attentions": distilbert_output.attentions,
        }

# Instantiate the custom model
num_labels = train_df['label'].nunique()

model = CustomDistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels,
    num_application_contexts=num_application_contexts,
    num_senders=num_senders
)

NameError: name 'train_dataset' is not defined

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    logging_steps=100,
    save_steps=500,
    report_to="none"
)


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

trainer.train()

Step,Training Loss
100,0.588932
200,0.078724
300,0.039098
400,0.075557
500,0.039803
600,0.030207
700,0.049361
800,0.033560
900,0.014262
1000,0.012426


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=5688, training_loss=0.024798022908877067, metrics={'train_runtime': 658.5962, 'train_samples_per_second': 69.074, 'train_steps_per_second': 8.637, 'total_flos': 1506605459337216.0, 'train_loss': 0.024798022908877067, 'epoch': 3.0})

First, let's say you want to save the model at the end of the current training session (or after an interruption). You can use `trainer.save_model()`:

In [ ]:
trainer.save_model("./results/final_model")
print("Model saved to ./results/final_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./results/final_model


To resume training from a checkpoint, you would typically follow these steps:

1.  Re-initialize your model with the same architecture.
2.  Re-initialize your tokenizer and datasets (as you've already done).
3.  Re-initialize your `TrainingArguments`.
4.  Re-initialize your `Trainer`, but this time, pass the `resume_from_checkpoint` argument pointing to the directory of your desired checkpoint (e.g., `./results/checkpoint-XXXX` or `./results/final_model` if you saved it).

In [ ]:
print("Resuming training from a saved checkpoint...")

# Assuming you want to resume from the latest checkpoint saved in './results'
# You might need to adjust this path based on the actual checkpoint folder name (e.g., './results/checkpoint-XXXX')
# Or if you saved a specific model like above, you can point to that directory.

# Re-initialize the model (it will load weights from the checkpoint)
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels
)

# Re-initialize the Trainer
resumed_trainer = Trainer(
    model=model,
    args=training_args, # Use the same training arguments
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

# Now, call train with resume_from_checkpoint
# You need to specify the path to the checkpoint directory.
# For example, if your checkpoints are saved in './results/checkpoint-1000',
# you would use resume_from_checkpoint="./results/checkpoint-1000"
# Or, if you want to resume from the last saved model:
# resumed_trainer.train(resume_from_checkpoint="./results/final_model")

# To resume from the *last* automatically saved checkpoint, you can usually just pass output_dir
resumed_trainer.train(resume_from_checkpoint=True) # Trainer will look for the latest checkpoint in output_dir


Resuming training from a saved checkpoint...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.b

Step,Training Loss


TrainOutput(global_step=5688, training_loss=0.0, metrics={'train_runtime': 0.0005, 'train_samples_per_second': 90644787.443, 'train_steps_per_second': 11333587.246, 'total_flos': 1506605459337216.0, 'train_loss': 0.0, 'epoch': 3.0})

In [ ]:
from google.colab import drive
import os

# Mount drive (only needed once per session)
drive.mount('/content/drive')

# Create project folder if it doesn't exist
PROJECT_PATH = "/content/drive/MyDrive/notification_model"
os.makedirs(PROJECT_PATH, exist_ok=True)

# Save model + tokenizer
SAVE_PATH = f"{PROJECT_PATH}/final_model"

trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("Model saved permanently to Google Drive.")
